# Milestone 2: Modern Deep Learning Architectures

## Question 1
Load train.csv using the Hugging Face datasets library. Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. What is the exact character length of the combined_text string for the row at index 51?

In [1]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files='../data/train.csv')['train']
dataset = dataset.map(lambda x: {'combined_text': str(x['prompt']) + " " + str(x['A'])})
print("Length of combined_text at index 51:", len(dataset[51]['combined_text']))


d:\mcp_solver\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 2000 examples [00:00, 35857.18 examples/s]
Map: 100%|██████████| 2000/2000 [00:00<00:00, 13597.34 examples/s]

Length of combined_text at index 51: 614


## Question 2
Initialize the bert-base-uncased tokenizer. What is the exact total vocabulary size hardcoded into this tokenizer?

In [2]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("Total vocabulary size:", tokenizer.vocab_size)


Total vocabulary size: 30522


## Question 3
Extract the exact integer ID assigned to the [SEP] (Separator) token using the bert-base-uncased tokenizer.

In [3]:
print("[SEP] token ID:", tokenizer.sep_token_id)


[SEP] token ID: 102


## Question 4
Tokenize the entire prompt column simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt'. What is the exact geometric shape of the resulting input_ids tensor?

In [4]:
prompts = [str(x) for x in dataset['prompt']]
tokens = tokenizer(prompts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
print("Shape of input_ids tensor:", tokens['input_ids'].shape)


Shape of input_ids tensor: torch.Size([2000, 128])


## Question 5
A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads. What is the exact dimensionality (size) of each individual attention head?

In [5]:
hidden_size = 768
num_attention_heads = 12
head_dim = hidden_size / num_attention_heads
print("Dimensionality of each attention head:", head_dim)


Dimensionality of each attention head: 64.0


## Question 6
Load the bert-base-uncased model. Tokenize the prompt from row ID 0 using default settings and pass it through the model. What is the exact shape of the last_hidden_state tensor returned?

In [6]:
from transformers import BertModel

model = BertModel.from_pretrained('bert-base-uncased')
tokens_0 = tokenizer(str(dataset[0]['prompt']), return_tensors='pt')
outputs_0 = model(**tokens_0)
print("Shape of last_hidden_state:", outputs_0.last_hidden_state.shape)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4905.22it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape of last_hidden_state: torch.Size([1, 31, 768])


## Question 7
Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token. What is the sum of the first 5 float values in this [CLS] vector? (Round to 4 decimal places).

In [7]:
cls_vector = outputs_0.last_hidden_state[0, 0, :]
sum_first_5 = cls_vector[:5].sum().item()
print("Sum of first 5 floats of CLS vector:", round(sum_first_5, 4))


Sum of first 5 floats of CLS vector: -1.2001


## Question 8
Load bert-base-uncased with output_attentions=True. Tokenize the exact string 'Light-ion fusion is a technique.' and pass it through the model. What is the exact attention weight that the [CLS] token pays to the word 'fusion' in the last layer, first attention head? (Round to 4 decimal places).

In [8]:
model_attn = BertModel.from_pretrained('bert-base-uncased', output_attentions=True)
tokens_fusion = tokenizer("Light-ion fusion is a technique.", return_tensors='pt')
outputs_attn = model_attn(**tokens_fusion)

attentions = outputs_attn.attentions
last_layer_attention = attentions[-1]

tokens_list = tokenizer.convert_ids_to_tokens(tokens_fusion['input_ids'][0])
fusion_idx = tokens_list.index('fusion')
cls_idx = 0

attn_weight = last_layer_attention[0, 0, cls_idx, fusion_idx]
print("Attention weight [CLS] pays to 'fusion':", round(attn_weight.item(), 4))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7219.67it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Attention weight [CLS] pays to 'fusion': 0.1025


## Question 9
Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Generate embeddings for the prompt and Option B for row ID 0. What is the cosine similarity between these two vectors using cos_sim()? (Round to 4 decimal places).

In [9]:
from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
prompt_emb = st_model.encode(str(dataset[0]['prompt']))
opt_b_emb = st_model.encode(str(dataset[0]['B']))

cos_sim = util.cos_sim(prompt_emb, opt_b_emb)
print("Cosine similarity MiniLM:", round(cos_sim.item(), 4))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2804.84it/s]


Cosine similarity MiniLM: 0.7658


## Question 10
Build two pipelines (TF-IDF vs MiniLM) ranking options using cosine similarity. What is the MAP@3 of the MiniLM pipeline? How many questions have the correct answer in the MiniLM Top-3 BUT NOT in the TF-IDF Top-3?

In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

df = pd.read_csv('../data/train.csv')

def tfidf_predict(row):
    docs = [str(row['prompt']), str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    vec = TfidfVectorizer()
    tfidf_matrix = vec.fit_transform(docs)
    sims = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    top_3_indices = np.argsort(sims)[-3:][::-1]
    options = ['A', 'B', 'C', 'D', 'E']
    return [options[i] for i in top_3_indices]

df['tfidf_top3'] = df.apply(tfidf_predict, axis=1)

def minilm_predict(row):
    docs = [str(row['prompt']), str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    embs = st_model.encode(docs)
    sims = util.cos_sim(embs[0:1], embs[1:]).flatten()
    sims_np = sims.numpy() if hasattr(sims, 'numpy') else sims
    top_3_indices = np.argsort(sims_np)[-3:][::-1]
    options = ['A', 'B', 'C', 'D', 'E']
    return [options[i] for i in top_3_indices]

df['minilm_top3'] = df.apply(minilm_predict, axis=1)

def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score

df['minilm_apk'] = df.apply(lambda x: apk(x['answer'], x['minilm_top3']), axis=1)
map3_minilm = df['minilm_apk'].mean()
print("MAP@3 MiniLM:", map3_minilm)

cond = df.apply(lambda x: (x['answer'] not in x['tfidf_top3']) and (x['answer'] in x['minilm_top3']), axis=1)
print("Count of questions MiniLM got right in top-3 that TF-IDF missed:", cond.sum())


MAP@3 MiniLM: 0.4230833333333333
Count of questions MiniLM got right in top-3 that TF-IDF missed: 564


## Question 11
Initialize zero-shot-classification pipeline with facebook/bart-large-mnli. For prompt of row index 1, pass Options A, B, C as candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [11]:
from transformers import pipeline

zs_pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_1 = dataset[1]
candidate_labels = [str(row_1['A']), str(row_1['B']), str(row_1['C'])]
result_11 = zs_pipe(str(row_1['prompt']), candidate_labels)

top_score_11 = result_11['scores'][0]
print("Zero-shot top probability:", round(top_score_11, 4))


Loading weights: 100%|██████████| 515/515 [00:00<00:00, 4759.17it/s]


Zero-shot top probability: 0.4575


## Question 12
Run the exact same zero-shot classification, passing multi_label=True. What is the absolute difference between the sum of probabilities in Q11 (Softmax) and Q12 (independent Sigmoids)?

In [12]:
result_12 = zs_pipe(str(row_1['prompt']), candidate_labels, multi_label=True)

sum_11 = sum(result_11['scores'])
sum_12 = sum(result_12['scores'])
print("Absolute difference in probability sums:", abs(sum_11 - sum_12))


Absolute difference in probability sums: 0.9994903962287935


## Question 13
Load google/flan-t5-small. Prompt it with 'Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B.' using row index 0. Set max_new_tokens=5. What is the exact string output?

In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_t5 = AutoTokenizer.from_pretrained("google/flan-t5-small")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row_0 = dataset[0]
prompt_13 = f"Question: {row_0['prompt']}. Is the correct answer A: {row_0['A']} or B: {row_0['B']}? Answer with just the letter A or B."

inputs = tokenizer_t5(prompt_13, return_tensors="pt")
outputs = model_t5.generate(**inputs, max_new_tokens=5)

output_str = tokenizer_t5.decode(outputs[0], skip_special_tokens=True)
print("Output string:", output_str)


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2408.41it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Output string: B
